# ETL y Limpieza de Datos Logísticos

Pipeline de extracción, diagnóstico, limpieza y estandarización del dataset `pedidos_logistica_raw.csv` para dejarlo listo para análisis exploratorio y visualización en BI.

In [2]:
import pandas as pd
import numpy as np

### 1. Carga de Datos y Diagnóstico Inicial

Se inspecciona la estructura general, valores nulos, filas completamente duplicadas y posibles `pedido_id` repetidos.

In [3]:
df = pd.read_csv("pedidos_logistica_raw.csv")
print(f"Registros cargados: {len(df)} | Columnas: {df.shape[1]}")
print("Filas completamente duplicadas:", df.duplicated().sum())
print("IDs de pedido duplicados:", df["pedido_id"].duplicated().sum())
print("\nDetalle de pedido_id repetidos (corresponden a órdenes distintas):")
print(df[df["pedido_id"].duplicated(keep=False)][["pedido_id", "fecha_pedido", "cliente", "monto_venta", "estado_entrega"]])

Registros cargados: 500 | Columnas: 9
Filas completamente duplicadas: 0
IDs de pedido duplicados: 3

Detalle de pedido_id repetidos (corresponden a órdenes distintas):
     pedido_id fecha_pedido              cliente monto_venta estado_entrega
14   ORD-10014   2026-06-24  Francisca Contreras    118090.0       Devuelto
15   ORD-10014   2026-08-27        Javiera Pérez     36340.0      Entregado
119  ORD-10119   2026-08-15       Isidora Torres     38150.0      Entregado
120  ORD-10119   2026-07-05      FRANCISCA SILVA    376040.0      Entregado
349  ORD-10349   2026-09-06  Francisca Rodríguez    431340.0      Entregado
350  ORD-10349   01/07/2026       matías muñoz       12530.0      Pendiente


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       500 non-null    str    
 1   fecha_pedido    500 non-null    str    
 2   fecha_entrega   351 non-null    str    
 3   cliente         486 non-null    str    
 4   region          500 non-null    str    
 5   categoria       489 non-null    str    
 6   monto_venta     479 non-null    str    
 7   costo_envio     467 non-null    float64
 8   estado_entrega  487 non-null    str    
dtypes: float64(1), str(8)
memory usage: 19.6 KB


### 2. Limpieza Financiera y Estandarización de Texto

* Se limpia `monto_venta` removiendo el símbolo `$`, sufijo `.0` y separadores de miles.
* Se normalizan `categoria` y `cliente` aplicando formato Título **solo sobre valores no nulos**, preservando los `NaN` reales.

In [5]:
df["monto_venta"] = (
    df["monto_venta"]
    .astype("string")
    .str.strip()
    .str.replace("$", "", regex=False)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(".", "", regex=False)
)
df["monto_venta"] = pd.to_numeric(df["monto_venta"], errors="coerce")

for col in ["categoria", "cliente"]:
    mask = df[col].notna()
    df.loc[mask, col] = df.loc[mask, col].astype(str).str.strip().str.title()

print(df[["pedido_id", "cliente", "categoria", "monto_venta"]].head(10))

   pedido_id            cliente  categoria  monto_venta
0  ORD-10000    Camila Martínez  Vestuario       -24280
1  ORD-10001  Valentina Fuentes    Calzado       371470
2  ORD-10002     Diego González    Belleza       119180
3  ORD-10003        Tomás Silva      Hogar       370400
4  ORD-10004    Nicolás Fuentes      Hogar       192680
5  ORD-10005     Francisca Díaz    Belleza       153130
6  ORD-10006   Joaquín Martínez      Hogar       157240
7  ORD-10007    Constanza Rojas    Belleza        96360
8  ORD-10008  Joaquín Contreras    Belleza       195780
9  ORD-10009     Benjamín Rojas  Vestuario       416580


### 3. Normalización de Regiones y Estado de Entrega

Se unifican variantes de regiones (`RM`, `Metropolitana`, `Valparaiso` sin tilde) y se estandariza `estado_entrega` manteniendo los nulos reales.

In [6]:
diccionario_regiones = {
    "rm": "Región Metropolitana",
    "metropolitana": "Región Metropolitana",
    "region metropolitana": "Región Metropolitana",
}
df["region"] = (
    df["region"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(diccionario_regiones)
    .str.title()
    .replace({"Valparaiso": "Valparaíso"})
)

mask_estado = df["estado_entrega"].notna()
df.loc[mask_estado, "estado_entrega"] = (
    df.loc[mask_estado, "estado_entrega"].astype(str).str.strip().str.title()
)

print("Regiones únicas:", sorted(df["region"].unique().tolist()))
print("Estados únicos:", df["estado_entrega"].unique().tolist())

Regiones únicas: ['Araucanía', 'Biobío', 'Coquimbo', 'Los Lagos', 'Maule', "O'Higgins", 'Región Metropolitana', 'Valparaíso']
Estados únicos: [nan, 'Devuelto', 'Entregado', 'En Tránsito', 'Cancelado', 'Pendiente']


### 4. Reglas de Negocio: Ingreso Total y Tipo de Movimiento

* Se separa explícitamente `Venta` (`monto_venta > 0`), `Devolución / Ajuste` (`monto_venta < 0`) y `Sin Monto` (`NaN`), evitando mezclar valores faltantes con devoluciones.
* Para calcular `ingreso_total` de las ventas positivas se usa `costo_envio.fillna(0)` para no perder el valor del producto cuando falta el costo de despacho.

In [7]:
es_venta = df["monto_venta"].gt(0).fillna(False)
es_ajuste = df["monto_venta"].lt(0).fillna(False)

df["ingreso_total"] = np.where(
    es_venta,
    df["monto_venta"].fillna(0) + df["costo_envio"].fillna(0),
    0,
)
df["tipo_movimiento"] = np.select(
    [es_venta, es_ajuste],
    ["Venta", "Devolución / Ajuste"],
    default="Sin Monto",
)

print(df["tipo_movimiento"].value_counts())
print(f"Ingreso bruto acumulado (ventas > 0): ${df['ingreso_total'].sum():,.0f}".replace(",", "."))

tipo_movimiento
Venta                  472
Sin Monto               21
Devolución / Ajuste      7
Name: count, dtype: int64
Ingreso bruto acumulado (ventas > 0): $109.102.840


### 5. Procesamiento de Fechas y Cálculo de Lead Time (Días de Entrega)

Se parsean formatos mixtos (`YYYY-MM-DD` y `DD/MM/YYYY`) y se crea `dias_entrega_limpio` aislando como `NaN` los 8 registros con fecha de entrega anterior a la fecha del pedido.

In [8]:
def parsear_fecha_segura(serie):
    """Convierte fechas en formato YYYY-MM-DD o DD/MM/YYYY de forma segura a datetime."""
    s = serie.astype("string").str.strip()
    mask_iso = s.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
    mask_dmy = s.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)

    fechas = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    fechas[mask_iso] = pd.to_datetime(s[mask_iso], format="%Y-%m-%d", errors="coerce")
    fechas[mask_dmy] = pd.to_datetime(s[mask_dmy], format="%d/%m/%Y", errors="coerce")
    return fechas

df["fecha_pedido"] = parsear_fecha_segura(df["fecha_pedido"])
df["fecha_entrega"] = parsear_fecha_segura(df["fecha_entrega"])
df["dias_entrega"] = (df["fecha_entrega"] - df["fecha_pedido"]).dt.days

df["fecha_pedido_str"] = df["fecha_pedido"].dt.strftime("%d/%m/%Y")
df["fecha_entrega_str"] = df["fecha_entrega"].dt.strftime("%d/%m/%Y")
df["dias_entrega_limpio"] = df["dias_entrega"].where(df["dias_entrega"] >= 0)

print("Registros con fecha de entrega válida (>= 0 días):", df["dias_entrega_limpio"].notna().sum())
print("Registros con lead time negativo (aislados):", (df["dias_entrega"] < 0).sum())
print("Registros sin fecha de entrega:", df["fecha_entrega"].isna().sum())
print(f"Lead time promedio: {df['dias_entrega_limpio'].mean():.2f} días | Mediana: {df['dias_entrega_limpio'].median():.1f} días")

Registros con fecha de entrega válida (>= 0 días): 343
Registros con lead time negativo (aislados): 8
Registros sin fecha de entrega: 149
Lead time promedio: 6.67 días | Mediana: 7.0 días


### 6. Conversión de Tipos y Exportación del Dataset Limpio

In [9]:
cols_enteras = ["monto_venta", "costo_envio", "ingreso_total", "dias_entrega_limpio"]
for col in cols_enteras:
    df[col] = df[col].round().astype("Int64")

df.to_csv("pedidos_logistica_limpios.csv", index=False, encoding="utf-8-sig")
print("Archivo 'pedidos_logistica_limpios.csv' exportado con éxito.")
print("\nNulos finales por columna:")
print(df.isna().sum())

Archivo 'pedidos_logistica_limpios.csv' exportado con éxito.

Nulos finales por columna:
pedido_id                0
fecha_pedido             0
fecha_entrega          149
cliente                 14
region                   0
categoria               11
monto_venta             21
costo_envio             33
estado_entrega          13
ingreso_total            0
tipo_movimiento          0
dias_entrega           149
fecha_pedido_str         0
fecha_entrega_str      149
dias_entrega_limpio    157
dtype: int64


### Conclusiones del Proceso ETL

* **Volumen y trazabilidad:** se procesaron **500 registros** y se exportó `pedidos_logistica_limpios.csv` con **15 columnas**. No hay filas completamente duplicadas (`0`), aunque se identificaron **3 `pedido_id` repetidos** (`ORD-10014`, `ORD-10119`, `ORD-10349`) que corresponden a órdenes distintas (distinta fecha, cliente y monto), por lo que se conservaron para auditoría.
* **Limpieza financiera:** `monto_venta` quedó normalizado en formato numérico entero (`Int64`). Se clasificaron **472** registros como `Venta`, **7** como `Devolución / Ajuste` (montos negativos) y **21** como `Sin Monto` (nulos reales). Utilizando `costo_envio.fillna(0)` en las ventas positivas, el ingreso bruto total registrado asciende a **109.102.840 CLP**.
* **Estandarización categórica:** se consolidaron **8 regiones** únicas, **6 categorías** (preservando 11 nulos reales), clientes normalizados (preservando 14 nulos reales) y **5 estados de entrega** (`Entregado`, `En Tránsito`, `Pendiente`, `Cancelado`, `Devuelto`, preservando 13 nulos reales).
* **Tiempos de entrega:** se obtuvieron **343 registros válidos** (`>= 0` días) con un **promedio de 6,67 días** y **mediana de 7,0 días**, aislando **8 casos con fechas invertidas** (lead time negativo) y **149 pedidos sin fecha de entrega**.